# Run multi agent
In this experiment, we will run the a multi agent on a couple of quesitons. We will not yet use the real model, but just experiment / demo using Groq. 

## Fetch model from groq

In [1]:
import os
import ast
import json
import regex

from time import sleep
import pandas as pd

from datasets import load_dataset

from groq import Groq

In [ ]:
with open("api_key.txt") as f:
    api_key = f.read().strip()
# client = Groq(
#     api_key=os.environ.get("GROQ_API_KEY"),
# )
client = Groq(
  api_key=api_key
)
model_name = "llama-3.3-70b-versatile"

# chat_completion = client.chat.completions.create(
#     messages=[
#         {
#             "role": "user",
#             "content": "Explain 2+2",
#         }
#     ],
#     model=model_name
# )

# print(chat_completion.choices[0].message.content)


'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'

In [3]:
from dataclasses import dataclass
import textwrap
@dataclass
class Role:
  name: str 
  behavior: str

  def instruction(self) -> str:
    return f"You are are {self.name}. {self.behavior}"
  def __str__(self):
    return f"Role: {self.name}\n{textwrap.fill(self.behavior, width=80)}"



In [4]:
Matematician = Role(
    name="Mathematician-and-problem-solver",
    behavior="You solve problems profficiently. You try to make every step in your reasoning clear and understanble, but also keeping it concise."
  )

print(Matematician)


Role: Mathematician-and-problem-solver
You solve problems profficiently. You try to make every step in your reasoning
clear and understanble, but also keeping it concise.


In [5]:
Verifier = Role(
  name="Verifier",
  behavior="You verfiy suggested solutions to problems. You look at all the steps, check if they make since, and suggest changes / constructive critisim. You are tough but fair. You are happy to discuss, byt you only accept a solution if you are 100% sure its correct. "
)
print(Verifier)

Role: Verifier
You verfiy suggested solutions to problems. You look at all the steps, check if
they make since, and suggest changes / constructive critisim. You are tough but
fair. You are happy to discuss, byt you only accept a solution if you are 100%
sure its correct.


In [20]:
from itertools import cycle
class Problem:
  roles_cycle: cycle
  all_roles: list[Role]
  problem_description: str
  welcome: str
  answer: str
  def __init__(self, roles: list[Role], problem_descr: str, answer: str):
    self.roles_cycle = cycle(roles)
    self.all_roles = roles
    self.problem_description = problem_descr
    last = self.all_roles[-1]
    others = ", ".join([x.name for x in self.all_roles[:-1]])  
    self.welcome = f"Welcome {others}, and {last.name}.\nTogether, you should solve the following problem: >>\n{problem_descr}.<<" 
    self.answer_format = "When you are done, you should submidt your answer as: ANSWER: <your answer>. No latex formatting, just the raw number/numbers or strings at the very end."

  def next_agent(self) -> Role: # this just loops over agents - can be changes to something smarter
    return next(self.roles_cycle) 
  
  def reset_cycle(self):
    self.roles_cycle = cycle(self.all_roles)
  
  def pose_problem(self):
    return f"{self.welcome}\n{self.answer_format}\n"
  
  def __str__(self) -> str:
    return textwrap.fill(self.pose_problem(), width=80)

In [21]:
twoplustwo=Problem(
  roles = [Matematician, Verifier],
  problem_descr="What is 2+2?",
  answer=2,
)
print(twoplustwo)

Welcome Mathematician-and-problem-solver, and Verifier. Together, you should
solve the following problem: >> What is 2+2?.<< When you are done, you should
submidt your answer as: ANSWER: <your answer>. No latex formatting, just the raw
number/numbers or strings at the very end.


## Creating a small dialog 


In [24]:
def conversation(n_steps: int, problem: Problem):
  problem.reset_cycle()
  roles = problem.all_roles
  messages =[
      {"role": "system", "content": role.instruction()} for role in roles
    ] + [{"role": "user", "content": problem.pose_problem()}]
  for step in range(n_steps):
    print(f"\nSTEP {step}: \n")
    next_agent = problem.next_agent()
    reply = client.chat.completions.create(
      model=model_name,
      messages=messages+[{"role": "user", "content": f"What do you say, {next_agent}"}] 
    ).choices[0].message.content
    print(f"{next_agent}: {reply}")
    messages.append({"role": "assistant", "content": reply})
    sleep(5)
  return messages
messages = conversation(2, twoplustwo)




STEP 0: 

Role: Mathematician-and-problem-solver
You solve problems profficiently. You try to make every step in your reasoning
clear and understanble, but also keeping it concise.: To solve the problem "What is 2+2?", I will follow a straightforward approach. 

1. I understand that the problem is asking for the sum of 2 and 2.
2. The sum of two numbers is the result of adding them together.
3. In basic arithmetic, 2 + 2 equals 4, as this is a fundamental addition fact.

Therefore, based on basic arithmetic principles, the answer to the problem "What is 2+2?" is 4.

Now, I will pass this solution to the Verifier for review and validation. 

Role: Verifier, please verify my solution.

STEP 1: 

Role: Verifier
You verfiy suggested solutions to problems. You look at all the steps, check if
they make since, and suggest changes / constructive critisim. You are tough but
fair. You are happy to discuss, byt you only accept a solution if you are 100%
sure its correct.: As the Verifier, I have

In [14]:
print("-------------------------------")
print("all messages")
for x in messages: print(x)

-------------------------------
all messages
{'role': 'system', 'content': 'You are are Mathematician-and-problem-solver. You solve problems profficiently. You try to make every step in your reasoning clear and understanble, but also keeping it concise.'}
{'role': 'system', 'content': 'You are are Verifier. You verfiy suggested solutions to problems. You look at all the steps, check if they make since, and suggest changes / constructive critisim. You are tough but fair. You are happy to discuss, byt you only accept a solution if you are 100% sure its correct. '}
{'role': 'user', 'content': 'Welcome Mathematician-and-problem-solver, and Verifier.\nTogether, you should solve the following problem: >>\nWhat is 2+2?.<<\nWhen you are done, you should submidt your answer as: ANSWER: <your answer>. No latex formatting, just the raw number/numbers or strings at the very end.\n'}
{'role': 'assistant', 'content': 'To solve the problem "What is 2+2?", I will follow basic arithmetic rules. The ope

## A harder problem

In [25]:
hard_problem = Problem(
  roles = [Matematician, Verifier],
  problem_descr=(
"""
Problem descr: 
Consider a $2025 \times 2025$ grid of unit squares. Matilda wishes to place on
the grid some rectangular tiles, possibly of different sizes, such that each
side of every tile lies on a grid line and every unit square is covered by at
most one tile. Determine the minimum number of tiles Matilda needs to place so
that each row and each column of the grid has exactly one unit square that is
not covered by any tile.

"""
  ),
  answer=2112
)
conversation(10, hard_problem)


STEP 0: 

Role: Mathematician-and-problem-solver
You solve problems profficiently. You try to make every step in your reasoning
clear and understanble, but also keeping it concise.: To solve this problem, let's break it down into steps. 

First, we need to understand the constraints: we have a 2025x2025 grid, and we want to place rectangular tiles such that each side of every tile lies on a grid line. Every unit square can be covered by at most one tile. 

The goal is to have exactly one unit square in each row and each column that is not covered by any tile. 

Let's consider the total number of unit squares that need to be left uncovered. Since there are 2025 rows and 2025 columns, and we need exactly one unit square in each row and each column to be uncovered, we might initially think we need 2025 + 2025 = 4050 unit squares to be left uncovered. However, this count includes the intersection point twice (the single square that is both in the uncovered row and column), so we actually 

[{'role': 'system',
  'content': 'You are are Mathematician-and-problem-solver. You solve problems profficiently. You try to make every step in your reasoning clear and understanble, but also keeping it concise.'},
 {'role': 'system',
  'content': 'You are are Verifier. You verfiy suggested solutions to problems. You look at all the steps, check if they make since, and suggest changes / constructive critisim. You are tough but fair. You are happy to discuss, byt you only accept a solution if you are 100% sure its correct. '},
 {'role': 'user',
  'content': 'Welcome Mathematician-and-problem-solver, and Verifier.\nTogether, you should solve the following problem: >>\n\nProblem descr: \nConsider a $2025 \times 2025$ grid of unit squares. Matilda wishes to place on\nthe grid some rectangular tiles, possibly of different sizes, such that each\nside of every tile lies on a grid line and every unit square is covered by at\nmost one tile. Determine the minimum number of tiles Matilda needs to